<a href="https://colab.research.google.com/github/DaisyXUUUUU/CIS-5450-Project/blob/main/notebooks/01_data_part_presentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 1 — Data Pipeline

**Owner:** Ziyue Xu (Daisy) · CIS 5450 · *Tipping Behavior in NYC Yellow Taxis*

Download → DuckDB sample → quality check → cleaning → zone & weather joins → features → time split → output.

| § | Topic |
| --- | --- |
| 0 | Question, target, leakage rule, setup |
| 1 | Acquisition |
| 2 | Sampling |
| 3 | Quality check |
| 4 | Cleaning |
| 5 | Joins |
| 6 | Features |
| 7 | Money fields |
| 8 | Time split |
| 9 | Output and data dictionary |

## 0. Question, target, leakage rule

- **Question:** among trips with a recorded card tip, which time, location, trip, and weather factors go with a high tip, and can it be predicted?
- **Target:** `high_tip = tip / fare ≥ 25%` (20% / 30% for sensitivity).
- **Prediction time:** trip has ended, rider has not paid yet.
- **Leakage rule:** `tip_amount`, `total_amount`, and anything built from them are never features.
- **Reproducibility:** `SEED = 5450` (the only random step), DuckDB 1.3.2.

In [ ]:
import os, sys, time, subprocess, tempfile
from pathlib import Path
try:
    import duckdb
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb==1.3.2"], check=True)
    import duckdb
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import requests
from IPython.display import display

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 80)
print("duckdb", duckdb.__version__, "| pandas", pd.__version__)

SEED = 5450
MONTHS = ["2026-04", "2026-05", "2026-06"]
N_PER_MONTH = 300_000
STUDY_START, STUDY_END = pd.Timestamp("2026-04-01"), pd.Timestamp("2026-07-01")
TRAIN_END, VAL_END = pd.Timestamp("2026-05-16"), pd.Timestamp("2026-06-01")
HIGH_TIP_THRESHOLD, SENSITIVITY_THRESHOLDS = 0.25, [0.20, 0.30]
RAIN_MM = 0.1
NYC_LAT, NYC_LON = 40.7128, -74.0060
JFK, LGA, EWR = 132, 138, 1
UNKNOWN_ZONES = [264, 265]

IN_COLAB = "google.colab" in sys.modules
FRESH_DIR = Path("/content/fresh") if IN_COLAB else Path(tempfile.gettempdir()) / "cis5450_fresh"
FRESH_DIR.mkdir(parents=True, exist_ok=True)
RAW = {m: FRESH_DIR / f"yellow_tripdata_{m}.parquet" for m in MONTHS}
ZONES_FILE, WEATHER_FILE = FRESH_DIR / "taxi_zone_lookup.csv", FRESH_DIR / "weather_hourly_nyc.csv"
OUTPUT_PATH = FRESH_DIR.parent / "output" / "trips_clean.parquet"

## 1. Acquisition

| Data | Source |
| --- | --- |
| Yellow taxi trips, Apr–Jun 2026 (monthly Parquet) | [NYC TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page) |
| Taxi zone lookup (265 zones) | same page |
| Hourly temperature and precipitation, Manhattan | [Open-Meteo Historical API](https://open-meteo.com/en/docs/historical-weather-api) |

- **Why Apr–Jun 2026:** recent, and all after the CBD fee started (2025-01-05), so fare rules are the same in every fold.

In [ ]:
TLC_BASE = os.environ.get("CIS5450_TLC_BASE", "https://d37ci6vzurychx.cloudfront.net")
SOURCES = {f"yellow_tripdata_{m}.parquet": f"{TLC_BASE}/trip-data/yellow_tripdata_{m}.parquet" for m in MONTHS}
SOURCES["taxi_zone_lookup.csv"] = f"{TLC_BASE}/misc/taxi_zone_lookup.csv"

def download(url, dest):
    with requests.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)

for name, url in SOURCES.items():
    if not (FRESH_DIR / name).exists():
        download(url, FRESH_DIR / name)

if not WEATHER_FILE.exists():
    resp = requests.get("https://archive-api.open-meteo.com/v1/archive", timeout=120, params={
        "latitude": NYC_LAT, "longitude": NYC_LON, "start_date": "2026-04-01", "end_date": "2026-06-30",
        "hourly": "temperature_2m,precipitation", "timezone": "America/New_York"})
    resp.raise_for_status()
    pd.DataFrame(resp.json()["hourly"]).to_csv(WEATHER_FILE, index=False)
print("Downloaded (UTC):", pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d %H:%M"))

con = duckdb.connect()
raw_summary = pd.DataFrame([{
    "month": m,
    "rows": pq.ParquetFile(RAW[m]).metadata.num_rows,
    "card_trips": con.execute(f"SELECT count(*) FROM read_parquet('{RAW[m]}') WHERE payment_type = 1").fetchone()[0],
} for m in MONTHS])
raw_summary["card_share"] = raw_summary["card_trips"] / raw_summary["rows"]
raw_summary.style.format({"rows": "{:,}", "card_trips": "{:,}", "card_share": "{:.1%}"})

- ~11.8M trips, 63–69% paid by card.
- **Card trips only:** TLC records tips only for card payments (cash tips show as 0).

## 2. Sampling (DuckDB)

- **Decision:** 300,000 card trips per month → 900,000, drawn uniformly with `SEED = 5450`.
- **How:** keep the trips with the smallest `hash(file_row_number, SEED)`; DuckDB reads Parquet directly, so raw rows never enter pandas.
- **Why not reservoir sampling:** TLC files are roughly time-sorted; in a trial, `USING SAMPLE reservoir` concentrated the sample in the first half of each month and left the May 16–31 validation fold with only 10,836 trips.
- **Check:** each day's share in the sample must be within 0.3 pp of the full month.

In [ ]:
parts = []
for m in MONTHS:
    part = con.execute(f'''
        SELECT * EXCLUDE (file_row_number)
        FROM read_parquet('{RAW[m]}', file_row_number = true)
        WHERE payment_type = 1
        ORDER BY hash(file_row_number, {SEED})
        LIMIT {N_PER_MONTH}
    ''').df()
    part["source_month"] = m
    parts.append(part)
sample = pd.concat(parts, ignore_index=True)
for c in ["tpep_pickup_datetime", "tpep_dropoff_datetime"]:
    sample[c] = pd.to_datetime(sample[c])
assert sample["source_month"].value_counts().eq(N_PER_MONTH).all()
print(f"Sample: {len(sample):,} trips")

# evenness check: share of trips per day, sample vs full month
files = ", ".join(repr(str(RAW[m])) for m in MONTHS)
pop = con.execute(f'''
    SELECT CAST(tpep_pickup_datetime AS DATE) AS day, count(*) AS n
    FROM read_parquet([{files}], union_by_name = true) WHERE payment_type = 1 GROUP BY ALL
''').df()
pop["day"] = pd.to_datetime(pop["day"])
samp = sample.assign(day=sample["tpep_pickup_datetime"].dt.normalize()).groupby(["source_month", "day"]).size()
gaps = {}
for m in MONTHS:
    p = pop[pop["day"].dt.strftime("%Y-%m") == m].set_index("day")["n"]
    q = samp.loc[m]
    q = q[q.index.strftime("%Y-%m") == m]
    gaps[m] = ((p / p.sum()).sub(q / q.sum(), fill_value=0).abs().max() * 100)
gaps = pd.Series(gaps, name="max daily gap (pp)").round(3)
display(gaps.to_frame())
assert (gaps < 0.3).all(), "sample is not spread evenly over the month"

## 3. Quality check (before cleaning)

Counts on the 900,000 sampled trips; one trip can have several problems.

In [ ]:
q = sample
dur = (q["tpep_dropoff_datetime"] - q["tpep_pickup_datetime"]).dt.total_seconds() / 60
display(q[["trip_distance", "fare_amount", "tip_amount", "total_amount", "passenger_count"]]
        .assign(duration_min=dur).describe(percentiles=[0.01, 0.5, 0.99]).T.round(2))

issues = pd.Series({
    "tip_amount = 0":                  (q["tip_amount"] == 0).sum(),
    "fare_amount <= 0":                (q["fare_amount"] <= 0).sum(),
    "pickup outside Apr-Jun":          (~q["tpep_pickup_datetime"].between(STUDY_START, STUDY_END, inclusive="left")).sum(),
    "duration < 1 or > 180 min":       (~dur.between(1, 180)).sum(),
    "distance = 0 or > 100 mi":        (~((q["trip_distance"] > 0) & (q["trip_distance"] <= 100))).sum(),
    "zone 264/265 (unknown)":          (q["PULocationID"].isin(UNKNOWN_ZONES) | q["DOLocationID"].isin(UNKNOWN_ZONES)).sum(),
    "RatecodeID = 99 (unknown)":       (q["RatecodeID"] == 99).sum(),
    "tip > fare":                      ((q["fare_amount"] > 0) & (q["tip_amount"] > q["fare_amount"])).sum(),
    "passenger_count = 0":             (q["passenger_count"] == 0).sum(),
    "request_source missing":          q["request_source"].isna().sum() if "request_source" in q else len(q),
}, name="trips").to_frame()
issues["pct"] = (issues["trips"] / len(q) * 100).round(2)
issues

| Issue | Action | Reason |
| --- | --- | --- |
| tip = 0 | drop | Can't tell "no tip" from "tipped in cash". |
| fare ≤ 0 | drop | Refunds / errors; tip ÷ fare undefined. |
| pickup outside Apr–Jun | drop | Wrong timestamps. |
| duration < 1 or > 180 min | drop | Meter errors / not a single trip. |
| distance 0 or > 100 mi | drop | GPS / meter errors. |
| zone 264 / 265 | drop | Unknown location, no borough. |
| RatecodeID 99 | drop | Unknown rate code. |
| tip > fare | drop | Mostly keying errors; extreme ratios. |
| passenger_count = 0 | keep trip, set to missing | Only the count is wrong; impute on train later. |
| `request_source` | drop column | Undocumented; empty for all card trips. |
| slow trips | **keep** | May be real traffic. |

Thresholds are fixed constants, not data quantiles, so test data does not shape any rule.

## 4. Cleaning

In [ ]:
RULES = [
    ("1. tip_amount > 0",               lambda d: d["tip_amount"] > 0),
    ("2. fare_amount > 0",              lambda d: d["fare_amount"] > 0),
    ("3. pickup within Apr-Jun 2026",   lambda d: d["tpep_pickup_datetime"].between(STUDY_START, STUDY_END, inclusive="left")),
    ("4. 1 <= duration <= 180 min",     lambda d: d["duration_min"].between(1, 180)),
    ("5. 0 < trip_distance <= 100 mi",  lambda d: (d["trip_distance"] > 0) & (d["trip_distance"] <= 100)),
    ("6. PU/DO zone not 264/265",       lambda d: ~d["PULocationID"].isin(UNKNOWN_ZONES) & ~d["DOLocationID"].isin(UNKNOWN_ZONES)),
    ("7. RatecodeID != 99",             lambda d: d["RatecodeID"].ne(99) | d["RatecodeID"].isna()),
    ("8. tip_pct <= 100%",              lambda d: d["tip_amount"] / d["fare_amount"] <= 1.0),
]
df = sample.drop(columns=["request_source"], errors="ignore").copy()
df["duration_min"] = (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60
log = [("0. sampled card trips", len(df), 0)]
for name, rule in RULES:
    before = len(df)
    df = df[rule(df).fillna(False).astype(bool)]
    log.append((name, len(df), before - len(df)))
df = df.reset_index(drop=True)
df.loc[df["passenger_count"] == 0, "passenger_count"] = np.nan

funnel = pd.DataFrame(log, columns=["step", "rows_after", "removed"])
funnel["pct_removed"] = funnel["removed"] / funnel.loc[0, "rows_after"] * 100
display(funnel.style.format({"rows_after": "{:,}", "removed": "{:,}", "pct_removed": "{:.2f}"}))
assert len(df) >= 50_000, "course requirement: at least 50,000 rows"

# why rule 7 removes nothing: RatecodeID 99 trips already fail an earlier rule
r99 = sample[sample["RatecodeID"] == 99].assign(
    duration_min=lambda d: (d["tpep_dropoff_datetime"] - d["tpep_pickup_datetime"]).dt.total_seconds() / 60)
first = pd.Series("passes rules 1-6", index=r99.index)
for name, rule in reversed(RULES[:6]):
    first[~rule(r99).fillna(False).astype(bool)] = name
print("RatecodeID 99 trips, first rule that removes them:")
print(first.value_counts().to_string())

- **Result:** 900,000 → 796,989 trips (88.6%); zero tips and bad durations are most of the removals.
- **Kept on purpose:** slow trips (short distance, long time) — likely traffic, no ground truth to call them errors.
- No imputation, capping, or scaling here; those are fitted on train only in modeling.

## 5. Joins

- **Zones:** LEFT JOIN pickup and drop-off IDs to the lookup (missing matches stay visible as nulls).
- **Weather:** join on pickup time floored to the hour = the last *completed* hour before pickup (no future data). All times are NYC local; no DST change in Apr–Jun.
- `is_rain = precip_mm ≥ 0.1`.

In [ ]:
zones = pd.read_csv(ZONES_FILE)
weather = pd.read_csv(WEATHER_FILE, parse_dates=["time"]).rename(
    columns={"time": "weather_hour", "temperature_2m": "temp_c", "precipitation": "precip_mm"})
assert len(weather) == (STUDY_END - STUDY_START) // pd.Timedelta(hours=1) and not weather["weather_hour"].duplicated().any()

n = len(df)
con.register("trips", df)
con.register("zones", zones)
df = con.execute('''
    SELECT t.*,
           pz.Borough AS PU_Borough, pz.Zone AS PU_Zone, pz.service_zone AS PU_service_zone,
           dz.Borough AS DO_Borough, dz.Zone AS DO_Zone, dz.service_zone AS DO_service_zone
    FROM trips t
    LEFT JOIN zones pz ON t.PULocationID = pz.LocationID
    LEFT JOIN zones dz ON t.DOLocationID = dz.LocationID
''').df()

df["pickup_hour_ts"] = df["tpep_pickup_datetime"].dt.floor("h")
con.register("trips_z", df)
con.register("weather", weather)
df = con.execute('''
    SELECT t.*, w.temp_c, w.precip_mm
    FROM trips_z t LEFT JOIN weather w ON t.pickup_hour_ts = w.weather_hour
''').df()
df["is_rain"] = (df["precip_mm"] >= RAIN_MM).astype("Int8").where(df["precip_mm"].notna())

assert len(df) == n, "a join changed the row count"
print(f"rows: {len(df):,} (unchanged)")
print(f"zone match:    {df['PU_Borough'].notna().mean():.1%} pickup, {df['DO_Borough'].notna().mean():.1%} drop-off")
print(f"weather match: {df['temp_c'].notna().mean():.1%}")
print(f"rainy-hour trips: {df['is_rain'].mean():.1%} | Manhattan pickups: {(df['PU_Borough'] == 'Manhattan').mean():.1%}")

- Both joins match 100% of trips with no row change; ~16% of trips are in rainy hours.
- ~91% of pickups are in Manhattan, so outer-borough results are less precise.

## 6. Features

All are functions of the trip's own record, zone, or the previous weather hour, so building them before the split cannot leak.

| Column | Definition |
| --- | --- |
| `pickup_hour`, `pickup_dow`, `is_weekend`, `is_late_night` | time of pickup (late night = 0–5h) |
| `avg_speed_mph` | 60 × distance / duration |
| `is_airport`, `is_newark` | JFK/LGA; Newark |
| `same_borough`, `has_cbd_fee`, `is_short_trip` | same borough; CBD fee; ≤ 2 mi |
| `is_rush_hour` | Mon–Fri 16–20h ($2.50 surcharge window); holidays kept, since meters still charged it |
| `is_night` | 20–6h ($1 surcharge window) |
| `tip_pct`, `high_tip`, `high_tip_20/30` | tip ÷ fare and targets — **not features** |
| `tip_pct_bill`, `high_tip_bill(_25/_30)` | tip ÷ (total − tip), above the % preset by > 1 cent — **not features** |
| `pickup_date` | cluster unit for SEs / bootstrap |

In [ ]:
pu = df["tpep_pickup_datetime"]
hour, dow = pu.dt.hour, pu.dt.dayofweek
df["pickup_hour"]   = hour.astype("int8")
df["pickup_dow"]    = dow.astype("int8")
df["is_weekend"]    = (dow >= 5).astype("int8")
df["is_late_night"] = (hour <= 5).astype("int8")
df["avg_speed_mph"] = 60 * df["trip_distance"] / df["duration_min"]
df["is_airport"]    = (df["PULocationID"].isin([JFK, LGA]) | df["DOLocationID"].isin([JFK, LGA])).astype("int8")
df["is_newark"]     = (df["PULocationID"].eq(EWR) | df["DOLocationID"].eq(EWR)).astype("int8")
df["same_borough"]  = (df["PU_Borough"] == df["DO_Borough"]).astype("int8")
df["has_cbd_fee"]   = (df["cbd_congestion_fee"].fillna(0) > 0).astype("int8")
df["is_short_trip"] = (df["trip_distance"] <= 2).astype("int8")

df["tip_pct"] = df["tip_amount"] / df["fare_amount"]
df["high_tip"] = (df["tip_pct"] >= HIGH_TIP_THRESHOLD).astype("int8")
for t in SENSITIVITY_THRESHOLDS:
    df[f"high_tip_{int(t * 100)}"] = (df["tip_pct"] >= t).astype("int8")
df["pickup_date"] = pu.dt.normalize()

# surcharge windows and bill-based tip (attached after `split` in Section 8)
bill = (df["total_amount"] - df["tip_amount"]).round(2)
extra_cols = {
    "is_rush_hour": ((dow <= 4) & hour.between(16, 19)).astype("int8"),
    "is_night": ((hour >= 20) | (hour <= 5)).astype("int8"),
    "tip_pct_bill": pd.Series(np.where(bill > 0, df["tip_amount"] / bill, np.nan), index=df.index),
}
for t, col in [(0.20, "high_tip_bill"), (0.25, "high_tip_bill_25"), (0.30, "high_tip_bill_30")]:
    s = (df["tip_amount"] > (t * bill).round(2) + 0.015).astype("int8")
    s[bill <= 0] = 0
    extra_cols[col] = s

# holiday check: did meters charge the rush surcharge on May 25 / June 19? (Vendor 2 only; see Section 7)
hol = pu.dt.normalize().isin(pd.to_datetime(["2026-05-25", "2026-06-19"]))
win = (dow <= 4) & hour.between(16, 19) & (df["VendorID"] == 2)
print(f"weekday 16-19h trips charged extra >= $2.50: ordinary days {(df.loc[win & ~hol, 'extra'] >= 2.5).mean():.0%}, "
      f"holidays {(df.loc[win & hol, 'extra'] >= 2.5).mean():.0%}")
pd.Series({"high_tip": df["high_tip"].mean(), "high_tip_bill": extra_cols["high_tip_bill"].mean(),
           "is_rush_hour": extra_cols["is_rush_hour"].mean(), "is_night": extra_cols["is_night"].mean()},
          name="share of trips").round(3).to_frame()

## 7. Money fields

- **Finding:** Vendor 1's `extra` already includes the congestion, CBD, and airport fees, which also have their own columns.
- **Decision:** bill = `total_amount − tip_amount` (no component sum needed). If `extra` is used as a feature, subtract those fees for Vendor 1 first.

In [ ]:
pre_tip = df["total_amount"] - df["tip_amount"]
base = df[["fare_amount", "extra", "mta_tax", "tolls_amount", "improvement_surcharge"]].sum(axis=1)
fees = df[["congestion_surcharge", "cbd_congestion_fee", "Airport_fee"]].fillna(0).sum(axis=1)
ok = lambda s: ((s - pre_tip).abs() <= 0.011).groupby(df["VendorID"]).mean()
pd.DataFrame({"trips": df["VendorID"].value_counts(),
              "components incl. 3 fees = bill": ok(base + fees),
              "components excl. 3 fees = bill": ok(base)}).sort_index().round(3)

## 8. Time split

| Fold | Dates | Use |
| --- | --- | --- |
| train | Apr 1 – May 15 | fit models and preprocessing |
| val | May 16 – 31 | tuning |
| test | June | final evaluation once |

- **Why by time:** same-day trips share weather and events; a random split would leak them and overstate performance.

In [ ]:
df["split"] = np.select([pu < TRAIN_END, pu < VAL_END], ["train", "val"], default="test")
for c, s in extra_cols.items():
    df[c] = s
df = df.sort_values("tpep_pickup_datetime").reset_index(drop=True)

split_summary = (df.groupby("split")
                   .agg(rows=("high_tip", "size"), days=("pickup_date", "nunique"),
                        high_tip_rate=("high_tip", "mean"), high_tip_bill_rate=("high_tip_bill", "mean"))
                   .reindex(["train", "val", "test"]))
assert split_summary["days"].tolist() == [45, 16, 30]
split_summary.style.format({"rows": "{:,}", "high_tip_rate": "{:.1%}", "high_tip_bill_rate": "{:.1%}"})

- Folds have similar high-tip rates, so none is skewed or too small.

## 9. Output and data dictionary

✅ feature · ⚠️ feature after a train-only fix · ❌ never a feature

In [ ]:
DICTIONARY = [
    # column, meaning, source, feature
    ("VendorID", "meter vendor: 1 = Creative Mobile, 2 = Curb", "TLC", "✅ categorical"),
    ("tpep_pickup_datetime", "pickup time (NYC local)", "TLC", "❌ use derived columns"),
    ("tpep_dropoff_datetime", "drop-off time", "TLC", "❌ use duration_min"),
    ("passenger_count", "passengers; 0 set to missing (3,047 trips)", "TLC", "⚠️ impute on train"),
    ("trip_distance", "miles, 0-100", "TLC", "✅"),
    ("RatecodeID", "1 standard, 2 JFK, 3 Newark, 4 Nassau/Westchester, 5 negotiated", "TLC", "✅ categorical"),
    ("store_and_fwd_flag", "Y = trip stored in the vehicle before upload", "TLC", "✅ (little information)"),
    ("PULocationID", "pickup taxi zone 1-263", "TLC", "✅ categorical"),
    ("DOLocationID", "drop-off taxi zone 1-263", "TLC", "✅ categorical"),
    ("payment_type", "always 1 (card)", "TLC", "❌ constant"),
    ("fare_amount", "time-and-distance fare, > 0; denominator of tip_pct", "TLC", "✅"),
    ("extra", "rush $2.50 / overnight $1; Vendor 1 also bundles congestion, CBD, airport fees", "TLC", "⚠️ de-duplicate Vendor 1"),
    ("mta_tax", "MTA tax, almost always $0.50", "TLC", "✅"),
    ("tip_amount", "card tip, > 0 and <= fare", "TLC", "❌ outcome"),
    ("tolls_amount", "tolls", "TLC", "✅"),
    ("improvement_surcharge", "improvement surcharge, almost always $1", "TLC", "✅"),
    ("total_amount", "total charged, INCLUDES the tip", "TLC", "❌ contains outcome"),
    ("congestion_surcharge", "NY State congestion surcharge, $0 or $2.50", "TLC", "✅"),
    ("Airport_fee", "JFK / LGA pickup fee", "TLC", "✅"),
    ("cbd_congestion_fee", "Manhattan CBD congestion fee, $0 or $0.75", "TLC", "✅ (or has_cbd_fee)"),
    ("source_month", "month file the trip was sampled from", "derived", "❌ bookkeeping"),
    ("duration_min", "drop-off minus pickup, 1-180 min", "derived", "✅"),
    ("PU_Borough", "pickup borough", "zone join", "✅ categorical"),
    ("PU_Zone", "pickup zone name", "zone join", "⚠️ duplicates PULocationID"),
    ("PU_service_zone", "Yellow Zone / Airports / Boro Zone / EWR", "zone join", "✅ categorical"),
    ("DO_Borough", "drop-off borough", "zone join", "✅ categorical"),
    ("DO_Zone", "drop-off zone name", "zone join", "⚠️ duplicates DOLocationID"),
    ("DO_service_zone", "as PU_service_zone, drop-off", "zone join", "✅ categorical"),
    ("pickup_hour_ts", "pickup time floored to the hour (weather join key)", "derived", "❌ join key"),
    ("temp_c", "temperature in the pickup hour (°C)", "weather join", "✅"),
    ("precip_mm", "precipitation in the hour ending at pickup_hour_ts (mm)", "weather join", "✅"),
    ("is_rain", "precip_mm >= 0.1", "derived", "✅"),
    ("pickup_hour", "pickup hour 0-23", "derived", "✅"),
    ("pickup_dow", "day of week, 0 = Monday", "derived", "✅"),
    ("is_weekend", "Saturday or Sunday", "derived", "✅"),
    ("is_late_night", "pickup 00:00-05:59 (subset of is_night)", "derived", "✅"),
    ("avg_speed_mph", "60 x distance / duration", "derived", "⚠️ cap chosen on train"),
    ("is_airport", "pickup or drop-off at JFK (132) / LGA (138)", "derived", "✅"),
    ("is_newark", "pickup or drop-off at Newark (1)", "derived", "✅"),
    ("same_borough", "pickup borough == drop-off borough", "derived", "✅"),
    ("has_cbd_fee", "cbd_congestion_fee > 0", "derived", "✅"),
    ("is_short_trip", "trip_distance <= 2 mi", "derived", "✅"),
    ("tip_pct", "tip_amount / fare_amount, <= 1", "derived", "❌ outcome"),
    ("high_tip", "tip_pct >= 0.25 (proposal target)", "derived", "❌ target"),
    ("high_tip_20", "tip_pct >= 0.20 (sensitivity)", "derived", "❌ target"),
    ("high_tip_30", "tip_pct >= 0.30 (sensitivity)", "derived", "❌ target"),
    ("pickup_date", "pickup date (cluster unit for SEs / bootstrap)", "derived", "❌ bookkeeping"),
    ("split", "train / val / test by pickup date", "derived", "❌ bookkeeping"),
    ("is_rush_hour", "Mon-Fri pickup 16:00-19:59, holidays included", "derived", "✅"),
    ("is_night", "pickup 20:00-05:59", "derived", "✅"),
    ("tip_pct_bill", "tip / (total_amount - tip_amount)", "derived", "❌ outcome"),
    ("high_tip_bill", "tip > round(0.20 x bill, 2) + 0.015", "derived", "❌ target"),
    ("high_tip_bill_25", "same rule at 25%", "derived", "❌ target"),
    ("high_tip_bill_30", "same rule at 30%", "derived", "❌ target"),
]

dictionary = (pd.DataFrame(DICTIONARY, columns=["column", "meaning", "source", "feature"])
                .set_index("column").reindex(df.columns).fillna("?"))
dictionary.insert(0, "dtype", df.dtypes.astype(str))
display(dictionary)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUTPUT_PATH, index=False)
print(f"Saved {len(df):,} rows x {df.shape[1]} columns -> {OUTPUT_PATH}")
con.close()

### Summary

- 11.8M raw trips → 900,000 card-trip sample (seed 5450, even across days) → **796,989** clean trips × 54 columns.
- Zone and weather joins: 100% matched; weather uses only the hour before pickup.
- Split by time: train / val / test = Apr 1–May 15 / May 16–31 / June.
- Leakage guard: tip and total (and anything from them) are never features; all fitted steps happen on train in modeling.
- **Limitations:** card tippers only; ~91% Manhattan; one weather point; sparse early weekday mornings.